# 01. Data Preprocessing & Dual-Horizon Feature Store
Tài liệu tiền xử lý và kỹ nghệ đặc trưng chuỗi thời gian cho 2 kịch bản:
- **Kịch bản 1 (1h Nowcasting - $t+1$):** Lags AQI ngắn hạn (`lag_0..3`, `roll_mean/std_3h`) + Thời tiết tại $t$.
- **Kịch bản 2 (24h Forecasting - $t+24$):** Thời tiết tương lai $W_{t+24}$ (`shift(-24)`) + Lags nền (`lag_0, 24, 48`, `roll_mean_24h`).
- **Phân chia dữ liệu:** Train (2018–2023), Validation (2024), Test (2025). Vùng đệm biên cắt bỏ 24h cuối để chống rò rỉ.


In [1]:
import os
import numpy as np
import pandas as pd

BASE_DIR = ".." if os.path.basename(os.getcwd()) == "notebooks" else "."
DATA_INPUT = os.path.join(BASE_DIR, "data", "raw", "california_aqi_model_ready.csv")
if not os.path.exists(DATA_INPUT):
    DATA_INPUT = os.path.join(BASE_DIR, "data", "california_aqi_model_ready.csv")
PROCESSED_DIR = os.path.join(BASE_DIR, "data", "processed")
if not os.path.exists(PROCESSED_DIR):
    PROCESSED_DIR = os.path.join(BASE_DIR, "data_processed")
os.makedirs(PROCESSED_DIR, exist_ok=True)

df_raw = pd.read_csv(DATA_INPUT)
print(f"Dataset shape: {df_raw.shape}")
df_raw.head()


Dataset shape: (181122, 21)


,time,source,station_id,station_name,lat,lon,pm25_ug_m3,pm10_ug_m3,target_aqi,target_type,...,relative_humidity_2m,rain,cloud_cover,wind_speed_10m,wind_direction_10m,surface_pressure,hour,dayofweek,month,year
0,2018-01-01 12:00:00,epa_aqs_station,FRES,Fresno - Garland,36.785380,-119.773210,101.2,136.0,183.035765,pm25_proxy,...,93.0,0.0,5.0,7.1,319.0,1010.4,12,0,1,2018
1,2018-01-01 12:00:00,epa_aqs_station,LA,Los Angeles - N. Main,34.066429,-118.226755,86.0,99.0,172.380544,pm25_proxy,...,70.0,0.0,0.0,7.7,62.0,1009.3,12,0,1,2018
2,2018-01-01 13:00:00,epa_aqs_station,FRES,Fresno - Garland,36.785380,-119.773210,101.4,135.5,183.175966,pm25_proxy,...,97.0,0.0,93.0,9.6,326.0,1010.5,13,0,1,2018
3,2018-01-01 13:00:00,epa_aqs_station,LA,Los Angeles - N. Main,34.066429,-118.226755,80.0,93.5,168.174535,pm25_proxy,...,67.0,0.0,0.0,6.5,93.0,1009.1,13,0,1,2018
4,2018-01-01 14:00:00,epa_aqs_station,FRES,Fresno - Garland,36.785380,-119.773210,101.6,135.0,183.316166,pm25_proxy,...,98.0,0.0,87.0,6.2,260.0,1010.8,14,0,1,2018


### 1. Vector Gió, Chỉ Số Khô Hạn VPD & Chu Kỳ Thời Gian
- Phân rã vector gió: $u = \text{wind\_speed} \times \cos(\text{rad})$, $v = \text{wind\_speed} \times \sin(\text{rad})$.
- Áp suất hơi nước thiếu hụt (Tetens formula):
  $$\text{VPD} = 0.6108 \times \exp\left(\frac{17.27 \times T}{T + 237.3}\right) \times \left(1 - \frac{\text{RH}}{100}\right)$$
- Chu kỳ $\sin/\cos$ cho giờ và tháng.


In [2]:
df_raw['time'] = pd.to_datetime(df_raw['time'])
df = df_raw.sort_values(['station_id', 'time']).reset_index(drop=True)

# 1. Vector gió (u, v)
rad = np.deg2rad(df['wind_direction_10m'])
df['wind_u'] = (df['wind_speed_10m'] * np.cos(rad)).astype(np.float32)
df['wind_v'] = (df['wind_speed_10m'] * np.sin(rad)).astype(np.float32)

# 2. Áp suất hơi nước thiếu hụt (VPD kpa)
T = df['temperature_2m']
RH = df['relative_humidity_2m']
es = 0.6108 * np.exp((17.27 * T) / (T + 237.3))
df['vpd_kpa'] = (es * (1.0 - RH / 100.0)).astype(np.float32)

# 3. Chu kỳ thời gian
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24.0).astype(np.float32)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24.0).astype(np.float32)
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12.0).astype(np.float32)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12.0).astype(np.float32)
df['is_weekend'] = df['dayofweek'].isin([5, 6]).astype(np.float32)

print("Hoàn tất tạo các biến vật lý và chu kỳ thời gian.")


Hoàn tất tạo các biến vật lý và chu kỳ thời gian.


### 2. Kỹ Nghệ Đặc Trưng Kịch Bản 1 (Short-term Nowcasting - 1h)


In [3]:
grouped = df.groupby('station_id')
df_1h = df.copy()

# Target t+1
df_1h['target'] = grouped['target_aqi'].shift(-1)

# AQI lags
df_1h['target_aqi_lag_0'] = df_1h['target_aqi']
df_1h['target_aqi_lag_1'] = grouped['target_aqi'].shift(1)
df_1h['target_aqi_lag_2'] = grouped['target_aqi'].shift(2)
df_1h['target_aqi_lag_3'] = grouped['target_aqi'].shift(3)
df_1h['diff_1h'] = df_1h['target_aqi_lag_0'] - df_1h['target_aqi_lag_1']

# Rolling thống kê
df_1h['roll_mean_3h'] = grouped['target_aqi'].transform(lambda s: s.shift(1).rolling(3).mean())
df_1h['roll_std_3h'] = grouped['target_aqi'].transform(lambda s: s.shift(1).rolling(3).std())

features_1h = [
    'station_id',
    'target_aqi_lag_0', 'target_aqi_lag_1', 'target_aqi_lag_2', 'target_aqi_lag_3',
    'diff_1h', 'roll_mean_3h', 'roll_std_3h',
    'temperature_2m', 'relative_humidity_2m', 'rain', 'cloud_cover',
    'wind_speed_10m', 'surface_pressure', 'wind_u', 'wind_v', 'vpd_kpa',
    'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'is_weekend'
]

cols_1h = ['time', 'year', 'target'] + features_1h
df_1h = df_1h[cols_1h].dropna().reset_index(drop=True)
print(f"Kịch bản 1h (sau dropna): {df_1h.shape}")


Kịch bản 1h (sau dropna): (181102, 25)


### 3. Kịch Bản 2 (Long-term Forecasting - 24h) với Dự Báo Thời Tiết Tương Lai $W_{t+24}$


In [4]:
df_24h = df.copy()
grouped = df_24h.groupby('station_id')

# Target t+24
df_24h['target'] = grouped['target_aqi'].shift(-24)

# Thời tiết tương lai W_{t+24}
df_24h['future_temp_2m'] = grouped['temperature_2m'].shift(-24)
df_24h['future_rh_2m'] = grouped['relative_humidity_2m'].shift(-24)
df_24h['future_wind_speed'] = grouped['wind_speed_10m'].shift(-24)
df_24h['future_wind_u'] = grouped['wind_u'].shift(-24)
df_24h['future_wind_v'] = grouped['wind_v'].shift(-24)
df_24h['future_rain'] = grouped['rain'].shift(-24)
df_24h['future_cloud_cover'] = grouped['cloud_cover'].shift(-24)
df_24h['future_surface_pressure'] = grouped['surface_pressure'].shift(-24)
df_24h['future_vpd_kpa'] = grouped['vpd_kpa'].shift(-24)

# Lags nền
df_24h['target_aqi_lag_0'] = df_24h['target_aqi']
df_24h['target_aqi_lag_24'] = grouped['target_aqi'].shift(24)
df_24h['target_aqi_lag_48'] = grouped['target_aqi'].shift(48)
df_24h['roll_mean_24h'] = grouped['target_aqi'].transform(lambda s: s.shift(24).rolling(24).mean())

# Thời gian tại t+24
df_24h['future_hour_sin'] = grouped['hour_sin'].shift(-24)
df_24h['future_hour_cos'] = grouped['hour_cos'].shift(-24)
df_24h['future_month_sin'] = grouped['month_sin'].shift(-24)
df_24h['future_month_cos'] = grouped['month_cos'].shift(-24)
df_24h['future_is_weekend'] = grouped['is_weekend'].shift(-24)

features_24h = [
    'station_id',
    'future_temp_2m', 'future_rh_2m', 'future_wind_speed',
    'future_wind_u', 'future_wind_v', 'future_rain',
    'future_cloud_cover', 'future_surface_pressure', 'future_vpd_kpa',
    'target_aqi_lag_0', 'target_aqi_lag_24', 'target_aqi_lag_48', 'roll_mean_24h',
    'future_hour_sin', 'future_hour_cos', 'future_month_sin', 'future_month_cos', 'future_is_weekend'
]

cols_24h = ['time', 'year', 'target'] + features_24h
df_24h = df_24h[cols_24h].dropna().reset_index(drop=True)
print(f"Kịch bản 24h (sau dropna): {df_24h.shape}")


Kịch bản 24h (sau dropna): (180762, 22)


### 4. Phân Chia Tập Dữ Liệu & Cắt Vùng Đệm Biên


In [5]:
def split_with_buffer(data, horizon_hours=24):
    train_mask = (data['year'] >= 2018) & (data['year'] <= 2023)
    val_mask = (data['year'] == 2024)
    test_mask = (data['year'] == 2025)
    
    df_train = data[train_mask].copy()
    df_val = data[val_mask].copy()
    df_test = data[test_mask].copy()
    
    if horizon_hours > 0:
        def trim_tail(df_sub):
            idx_keep = []
            for _, grp in df_sub.groupby('station_id'):
                if len(grp) > horizon_hours:
                    idx_keep.extend(grp.index[:-horizon_hours])
                else:
                    idx_keep.extend(grp.index)
            return df_sub.loc[idx_keep]
            
        df_train = trim_tail(df_train).reset_index(drop=True)
        df_val = trim_tail(df_val).reset_index(drop=True)
        
    return df_train, df_val, df_test

train_1h, val_1h, test_1h = split_with_buffer(df_1h, horizon_hours=1)
print(f"[1h] Train: {train_1h.shape}, Val 2024: {val_1h.shape}, Test 2025: {test_1h.shape}")

train_24h, val_24h, test_24h = split_with_buffer(df_24h, horizon_hours=24)
print(f"[24h] Train: {train_24h.shape}, Val 2024: {val_24h.shape}, Test 2025: {test_24h.shape}")


[1h] Train: (111073, 25), Val 2024: (43727, 25), Test 2025: (26292, 25)


[24h] Train: (110733, 22), Val 2024: (43582, 22), Test 2025: (26207, 22)


### 5. Xuất Dữ Liệu Ra `data_processed/`


In [6]:
def export_df(df_save, base_name):
    p_path = os.path.join(PROCESSED_DIR, f"{base_name}.parquet")
    c_path = os.path.join(PROCESSED_DIR, f"{base_name}.csv")
    try:
        df_save.to_parquet(p_path, index=False)
        print(f"Exported: {p_path}")
    except Exception:
        df_save.to_csv(c_path, index=False)
        print(f"Exported: {c_path}")

export_df(train_1h, "dataset_1h_train")
export_df(val_1h, "dataset_1h_val")
export_df(test_1h, "dataset_1h_test")

export_df(train_24h, "dataset_24h_train")
export_df(val_24h, "dataset_24h_val")
export_df(test_24h, "dataset_24h_test")


Exported: .\data\processed\dataset_1h_train.parquet


Exported: .\data\processed\dataset_1h_val.parquet


Exported: .\data\processed\dataset_1h_test.parquet
Exported: .\data\processed\dataset_24h_train.parquet


Exported: .\data\processed\dataset_24h_val.parquet
Exported: .\data\processed\dataset_24h_test.parquet
